# TLSAN: Time-aware Long- and Short-term Attention Network
## For Next-Item Recommendation

---

## 1. Problem Statement

Given a user's interaction sequence:

$$S_u = \{(v_1, t_1), (v_2, t_2), \ldots, (v_n, t_n)\}$$

where $$v_i$$ is the item and $$t_i$$ is the timestamp, predict the next item $$v_{n+1}$$ the user will interact with.

**Key challenges:**
- Users have both **long-term stable preferences** and **short-term dynamic interests**
- **Temporal patterns** (time intervals between interactions) carry predictive signal
- Traditional attention treats all history uniformly, ignoring temporal dynamics

## 2. Architecture Overview

TLSAN decomposes the problem into three modules:

```
Input Sequence → [Embedding + Time Encoding] → [Long-term Attention] → Long-term Representation
                                              → [Short-term Attention] → Short-term Representation
                                              → [Fusion Gate] → Final Prediction
```

| Component | Purpose |
| --- | --- |
| Item Embedding | Dense vector representation of items |
| Time Encoding | Captures temporal intervals between interactions |
| Long-term Attention | Models stable preferences over full history |
| Short-term Attention | Models recent dynamic interests |
| Gated Fusion | Adaptively combines long/short-term signals |

## 3. Mathematical Formulation

### Step 1: Item Embedding + Positional Encoding

Each item $$v_i$$ is mapped to a dense vector:

$$\mathbf{e}_i = \mathbf{E}[v_i] \in \mathbb{R}^d$$

where $$\mathbf{E} \in \mathbb{R}^{|V| \times d}$$ is the item embedding matrix.

**Positional encoding** captures sequence order:

$$\mathbf{p}_i = \text{PE}(i) \in \mathbb{R}^d$$

Using sinusoidal encoding:

$$\text{PE}(i, 2k) = \sin\left(\frac{i}{10000^{2k/d}}\right), \quad \text{PE}(i, 2k+1) = \cos\left(\frac{i}{10000^{2k/d}}\right)$$

### Step 2: Time Interval Encoding

The time gap between consecutive interactions encodes temporal dynamics:

$$\Delta t_i = t_i - t_{i-1}$$

This is projected into embedding space via a learnable kernel:

$$\mathbf{z}_i = \mathbf{W}_t \cdot \phi(\Delta t_i) + \mathbf{b}_t$$

where $$\phi(\cdot)$$ is a time kernel function. Common choices:

- **Linear**: $$\phi(\Delta t) = \Delta t$$
- **Exponential decay**: $$\phi(\Delta t) = e^{-\Delta t / \tau}$$
- **Periodic**: $$\phi(\Delta t) = [\sin(\omega_1 \Delta t), \cos(\omega_1 \Delta t), \ldots]$$

The final input representation combines all three:

$$\mathbf{h}_i = \mathbf{e}_i + \mathbf{p}_i + \mathbf{z}_i$$

### Step 3: Long-term Attention (Global User Preference)

The long-term module applies **time-aware multi-head self-attention** over the entire sequence:

$$\mathbf{H} = [\mathbf{h}_1, \ldots, \mathbf{h}_n]$$

**Standard scaled dot-product attention:**

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\right)\mathbf{V}$$

**Time-aware modification** — inject temporal bias into attention scores:

$$\alpha_{ij} = \frac{\mathbf{q}_i \cdot \mathbf{k}_j}{\sqrt{d_k}} + \beta \cdot f(\Delta t_{ij})$$

where $$f(\Delta t_{ij})$$ is a temporal decay function and $$\beta$$ is a learnable scale. The decay function penalizes interactions that are far apart in time:

$$f(\Delta t_{ij}) = -\frac{|t_i - t_j|}{\tau}$$

**Multi-head formulation:**

$$\mathbf{Q} = \mathbf{H}\mathbf{W}_Q^{(h)}, \quad \mathbf{K} = \mathbf{H}\mathbf{W}_K^{(h)}, \quad \mathbf{V} = \mathbf{H}\mathbf{W}_V^{(h)}$$

$$\text{MultiHead}(\mathbf{H}) = \text{Concat}(\text{head}_1, \ldots, \text{head}_H)\mathbf{W}_O$$

Followed by a feed-forward network with residual connection:

$$\mathbf{H}^{(l)} = \text{LayerNorm}(\mathbf{H}^{(l-1)} + \text{MultiHead}(\mathbf{H}^{(l-1)}))$$

$$\mathbf{H}^{(l)} = \text{LayerNorm}(\mathbf{H}^{(l)} + \text{FFN}(\mathbf{H}^{(l)}))$$

The **long-term representation** is obtained via target-aware attention pooling:

$$\mathbf{s}_{\text{long}} = \sum_{i=1}^{n} \alpha_i \cdot \mathbf{h}_i^{(L)}, \quad \alpha_i = \frac{\exp(\mathbf{w}^\top \mathbf{h}_i^{(L)})}{\sum_j \exp(\mathbf{w}^\top \mathbf{h}_j^{(L)})}$$

### Step 4: Short-term Attention (Recent Dynamic Interest)

The short-term module operates on the **last $$k$$ items** of the sequence:

$$\mathbf{H}_{\text{short}} = [\mathbf{h}_{n-k+1}, \ldots, \mathbf{h}_n]$$

It uses a **local time-aware attention** mechanism:

$$\mathbf{s}_{\text{short}} = \sum_{i=n-k+1}^{n} \gamma_i \cdot \mathbf{h}_i$$

where the attention weights incorporate recency:

$$\gamma_i = \text{softmax}\left(\frac{(\mathbf{W}_q \mathbf{h}_n)^\top (\mathbf{W}_k \mathbf{h}_i)}{\sqrt{d_k}} + \lambda \cdot g(t_n - t_i)\right)$$

Here $$\mathbf{h}_n$$ (the last item) serves as the query, attending over recent items. The function $$g(\cdot)$$ provides an exponential recency boost:

$$g(\Delta t) = \exp(-\Delta t / \tau_{\text{short}})$$

This ensures the most recent interactions receive higher attention, modulated by temporal proximity.

### Step 5: Gated Fusion

A gating mechanism adaptively combines the two representations:

$$\mathbf{g} = \sigma(\mathbf{W}_g [\mathbf{s}_{\text{long}} \| \mathbf{s}_{\text{short}}] + \mathbf{b}_g)$$

$$\mathbf{s}_u = \mathbf{g} \odot \mathbf{s}_{\text{long}} + (1 - \mathbf{g}) \odot \mathbf{s}_{\text{short}}$$

where $$\sigma$$ is the sigmoid function, $$\|$$ denotes concatenation, and $$\odot$$ is element-wise multiplication.

### Step 6: Prediction

The final prediction score for candidate item $$v$$ is:

$$\hat{y}_{u,v} = \mathbf{s}_u^\top \mathbf{e}_v$$

Trained with cross-entropy loss:

$$\mathcal{L} = -\sum_{(u, v^+)} \log \sigma(\hat{y}_{u,v^+}) - \sum_{(u, v^-)} \log(1 - \sigma(\hat{y}_{u,v^-}))$$

In [0]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class TimeEncoding(nn.Module):
    """Encodes time intervals into dense vectors using periodic kernels."""

    def __init__(self, d_model: int, max_period: float = 10000.0):
        super().__init__()
        self.d_model = d_model
        # Learnable linear projection of time features
        self.time_proj = nn.Linear(d_model, d_model)
        # Frequency basis for periodic encoding
        freqs = torch.exp(
            torch.arange(0, d_model, 2).float() * -(math.log(max_period) / d_model)
        )
        self.register_buffer("freqs", freqs)

    def forward(self, delta_t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            delta_t: (batch, seq_len) time intervals
        Returns:
            (batch, seq_len, d_model) time embeddings
        """
        # Expand for broadcasting: (batch, seq_len, d_model//2)
        t = delta_t.unsqueeze(-1) * self.freqs
        # Interleave sin/cos: (batch, seq_len, d_model)
        time_enc = torch.zeros(*delta_t.shape, self.d_model, device=delta_t.device)
        time_enc[..., 0::2] = torch.sin(t)
        time_enc[..., 1::2] = torch.cos(t)
        return self.time_proj(time_enc)


class TimeAwareMultiHeadAttention(nn.Module):
    """Multi-head attention with temporal bias in attention scores."""

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

        # Learnable temporal decay scale per head
        self.beta = nn.Parameter(torch.ones(n_heads, 1, 1) * 0.1)
        self.tau = nn.Parameter(torch.ones(1) * 1.0)  # time scale

        self.dropout = nn.Dropout(dropout)

    def forward(
        self, H: torch.Tensor, time_matrix: torch.Tensor, mask: torch.Tensor = None
    ) -> torch.Tensor:
        """
        Args:
            H: (batch, seq_len, d_model) input representations
            time_matrix: (batch, seq_len, seq_len) pairwise |t_i - t_j|
            mask: (batch, 1, seq_len) padding mask
        Returns:
            (batch, seq_len, d_model) attended output
        """
        B, N, _ = H.shape

        # Linear projections -> (batch, n_heads, seq_len, d_k)
        Q = self.W_Q(H).view(B, N, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(H).view(B, N, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(H).view(B, N, self.n_heads, self.d_k).transpose(1, 2)

        # Scaled dot-product: (batch, n_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Time-aware bias: f(delta_t) = -|t_i - t_j| / tau
        # time_matrix: (batch, seq_len, seq_len) -> (batch, 1, seq_len, seq_len)
        time_bias = -time_matrix.unsqueeze(1) / (self.tau + 1e-8)
        scores = scores + self.beta * time_bias

        # Apply causal + padding mask
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(1) == 0, -1e9)

        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        # Weighted sum: (batch, n_heads, seq_len, d_k)
        context = torch.matmul(attn, V)
        # Concat heads: (batch, seq_len, d_model)
        context = context.transpose(1, 2).contiguous().view(B, N, -1)
        return self.W_O(context)


class TransformerBlock(nn.Module):
    """Single transformer block with time-aware attention + FFN."""

    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.attention = TimeAwareMultiHeadAttention(d_model, n_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(
        self, H: torch.Tensor, time_matrix: torch.Tensor, mask: torch.Tensor = None
    ) -> torch.Tensor:
        # Pre-norm residual connection
        H = H + self.attention(self.norm1(H), time_matrix, mask)
        H = H + self.ffn(self.norm2(H))
        return H


print("Core modules defined: TimeEncoding, TimeAwareMultiHeadAttention, TransformerBlock")

Core modules defined: TimeEncoding, TimeAwareMultiHeadAttention, TransformerBlock


In [0]:
class TLSAN(nn.Module):
    """
    Time-aware Long- and Short-term Attention Network.

    Architecture:
        Input -> Embedding + Position + Time Encoding
             -> Long-term: Stacked time-aware transformer blocks over full sequence
             -> Short-term: Local attention over last k items
             -> Gated fusion -> Prediction
    """

    def __init__(
        self,
        n_items: int,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        d_ff: int = 256,
        max_seq_len: int = 50,
        short_term_k: int = 5,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.d_model = d_model
        self.short_term_k = short_term_k
        self.max_seq_len = max_seq_len

        # --- Embedding Layers ---
        self.item_embedding = nn.Embedding(n_items + 1, d_model, padding_idx=0)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        self.time_encoding = TimeEncoding(d_model)
        self.embed_dropout = nn.Dropout(dropout)

        # --- Long-term Module: Stacked transformer blocks ---
        self.long_term_blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        # Attention pooling for long-term summary
        self.long_pool_weight = nn.Linear(d_model, 1)

        # --- Short-term Module: Local attention ---
        self.short_W_q = nn.Linear(d_model, d_model)
        self.short_W_k = nn.Linear(d_model, d_model)
        self.short_tau = nn.Parameter(torch.ones(1) * 1.0)
        self.short_lambda = nn.Parameter(torch.ones(1) * 0.1)

        # --- Gated Fusion ---
        self.gate = nn.Linear(2 * d_model, d_model)

        # --- Output ---
        self.output_norm = nn.LayerNorm(d_model)

    def _build_time_matrix(self, timestamps: torch.Tensor) -> torch.Tensor:
        """Compute pairwise absolute time differences.
        Args:
            timestamps: (batch, seq_len)
        Returns:
            (batch, seq_len, seq_len) matrix of |t_i - t_j|
        """
        # (batch, seq_len, 1) - (batch, 1, seq_len) -> (batch, seq_len, seq_len)
        return torch.abs(timestamps.unsqueeze(-1) - timestamps.unsqueeze(-2))

    def _compute_delta_t(self, timestamps: torch.Tensor) -> torch.Tensor:
        """Compute consecutive time intervals: delta_t_i = t_i - t_{i-1}."""
        delta = torch.zeros_like(timestamps)
        delta[:, 1:] = timestamps[:, 1:] - timestamps[:, :-1]
        return delta

    def forward(
        self,
        item_seq: torch.Tensor,
        time_seq: torch.Tensor,
        mask: torch.Tensor = None,
    ) -> torch.Tensor:
        """
        Args:
            item_seq: (batch, seq_len) item indices
            time_seq: (batch, seq_len) timestamps (normalized)
            mask: (batch, seq_len) binary mask (1=valid, 0=pad)
        Returns:
            (batch, d_model) user representation for prediction
        """
        B, N = item_seq.shape

        # === Step 1: Embedding + Position + Time ===
        positions = torch.arange(N, device=item_seq.device).unsqueeze(0).expand(B, -1)
        delta_t = self._compute_delta_t(time_seq)

        e = self.item_embedding(item_seq)          # (B, N, d)
        p = self.position_embedding(positions)      # (B, N, d)
        z = self.time_encoding(delta_t)             # (B, N, d)

        H = self.embed_dropout(e + p + z)           # h_i = e_i + p_i + z_i

        # Build pairwise time matrix for attention bias
        time_matrix = self._build_time_matrix(time_seq)

        # Attention mask: (batch, 1, seq_len) for broadcasting
        attn_mask = mask.unsqueeze(1) if mask is not None else None

        # === Step 3: Long-term Attention ===
        H_long = H
        for block in self.long_term_blocks:
            H_long = block(H_long, time_matrix, attn_mask)

        # Attention pooling: s_long = sum(alpha_i * h_i^L)
        pool_scores = self.long_pool_weight(H_long).squeeze(-1)  # (B, N)
        if mask is not None:
            pool_scores = pool_scores.masked_fill(mask == 0, -1e9)
        pool_alpha = F.softmax(pool_scores, dim=-1)  # (B, N)
        s_long = torch.bmm(pool_alpha.unsqueeze(1), H_long).squeeze(1)  # (B, d)

        # === Step 4: Short-term Attention ===
        # Take last k items
        k = min(self.short_term_k, N)
        H_short = H[:, -k:, :]            # (B, k, d)
        t_short = time_seq[:, -k:]         # (B, k)

        # Query = last item representation
        query = self.short_W_q(H[:, -1:, :])     # (B, 1, d)
        keys = self.short_W_k(H_short)            # (B, k, d)

        # Attention scores with recency bias
        short_scores = torch.bmm(query, keys.transpose(1, 2)) / math.sqrt(self.d_model)
        # g(delta_t) = exp(-delta_t / tau_short)
        recency = torch.exp(-(time_seq[:, -1:] - t_short) / (self.short_tau + 1e-8))
        short_scores = short_scores.squeeze(1) + self.short_lambda * recency  # (B, k)

        gamma = F.softmax(short_scores, dim=-1)  # (B, k)
        s_short = torch.bmm(gamma.unsqueeze(1), H_short).squeeze(1)  # (B, d)

        # === Step 5: Gated Fusion ===
        gate_input = torch.cat([s_long, s_short], dim=-1)  # (B, 2d)
        g = torch.sigmoid(self.gate(gate_input))            # (B, d)
        s_u = g * s_long + (1 - g) * s_short                # (B, d)

        return self.output_norm(s_u)

    def predict(self, user_repr: torch.Tensor, candidate_items: torch.Tensor) -> torch.Tensor:
        """
        Compute prediction scores: y_hat = s_u^T * e_v
        Args:
            user_repr: (batch, d_model)
            candidate_items: (batch, n_candidates) or (n_items,)
        Returns:
            (batch, n_candidates) scores
        """
        item_embs = self.item_embedding(candidate_items)  # (..., d)
        if item_embs.dim() == 2:
            # All items: (n_items, d) -> scores = (batch, n_items)
            return torch.matmul(user_repr, item_embs.T)
        else:
            # Per-sample candidates: (batch, n_cand, d)
            return torch.bmm(item_embs, user_repr.unsqueeze(-1)).squeeze(-1)


print("TLSAN model class defined.")
print(f"\nModel architecture:")
print(f"  - Time-aware multi-head self-attention (long-term)")
print(f"  - Local recency-boosted attention (short-term)")
print(f"  - Sigmoid gated fusion")
print(f"  - Dot-product prediction")

TLSAN model class defined.

Model architecture:
  - Time-aware multi-head self-attention (long-term)
  - Local recency-boosted attention (short-term)
  - Sigmoid gated fusion
  - Dot-product prediction


In [0]:
class TLSANTrainer:
    """Training wrapper with BPR/CE loss and evaluation."""

    def __init__(self, model: TLSAN, lr: float = 1e-3, weight_decay: float = 1e-5):
        self.model = model
        self.optimizer = torch.optim.Adam(
            model.parameters(), lr=lr, weight_decay=weight_decay
        )
        self.scheduler = torch.optim.lr_scheduler.StepLR(
            self.optimizer, step_size=10, gamma=0.5
        )

    def compute_loss(
        self,
        user_repr: torch.Tensor,
        pos_items: torch.Tensor,
        neg_items: torch.Tensor,
    ) -> torch.Tensor:
        """
        Binary cross-entropy loss:
        L = -log(sigma(y_pos)) - log(1 - sigma(y_neg))
        """
        pos_emb = self.model.item_embedding(pos_items)  # (B, d)
        neg_emb = self.model.item_embedding(neg_items)  # (B, d)

        pos_score = (user_repr * pos_emb).sum(dim=-1)   # (B,)
        neg_score = (user_repr * neg_emb).sum(dim=-1)   # (B,)

        loss = -torch.log(torch.sigmoid(pos_score) + 1e-8).mean() \
               - torch.log(1 - torch.sigmoid(neg_score) + 1e-8).mean()
        return loss

    def train_step(
        self,
        item_seq: torch.Tensor,
        time_seq: torch.Tensor,
        mask: torch.Tensor,
        pos_items: torch.Tensor,
        neg_items: torch.Tensor,
    ) -> float:
        self.model.train()
        self.optimizer.zero_grad()

        user_repr = self.model(item_seq, time_seq, mask)
        loss = self.compute_loss(user_repr, pos_items, neg_items)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=5.0)
        self.optimizer.step()

        return loss.item()


# --- Demo: Instantiate and run a forward pass ---
torch.manual_seed(42)

# Hyperparameters
N_ITEMS = 10000
D_MODEL = 64
N_HEADS = 4
N_LAYERS = 2
MAX_SEQ_LEN = 50
SHORT_K = 5
BATCH_SIZE = 4
SEQ_LEN = 20

# Create model
model = TLSAN(
    n_items=N_ITEMS,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    max_seq_len=MAX_SEQ_LEN,
    short_term_k=SHORT_K,
)

# Synthetic input
item_seq = torch.randint(1, N_ITEMS, (BATCH_SIZE, SEQ_LEN))
time_seq = torch.sort(torch.rand(BATCH_SIZE, SEQ_LEN) * 100, dim=1)[0]  # sorted timestamps
mask = torch.ones(BATCH_SIZE, SEQ_LEN)
mask[:, -3:] = 0  # simulate some padding

# Forward pass
user_repr = model(item_seq, time_seq, mask)
print(f"User representation shape: {user_repr.shape}")  # (4, 64)

# Predict over all items
all_items = torch.arange(1, N_ITEMS + 1)
scores = model.predict(user_repr, all_items)
print(f"Prediction scores shape: {scores.shape}")  # (4, 10000)
print(f"Top-5 recommended items for user 0: {scores[0].topk(5).indices.tolist()}")

# Training step demo
trainer = TLSANTrainer(model)
pos_items = torch.randint(1, N_ITEMS, (BATCH_SIZE,))
neg_items = torch.randint(1, N_ITEMS, (BATCH_SIZE,))
loss = trainer.train_step(item_seq, time_seq, mask, pos_items, neg_items)
print(f"\nTraining loss (1 step): {loss:.4f}")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

User representation shape: torch.Size([4, 64])
Prediction scores shape: torch.Size([4, 10000])
Top-5 recommended items for user 0: [4623, 5143, 7706, 1804, 1871]

Training loss (1 step): 8.1453
Total parameters: 764,173


## 4. Summary of Key Design Choices

| Design Choice | Mathematical Mechanism | Benefit |
| --- | --- | --- |
| Time interval encoding | $$\mathbf{z}_i = W_t \cdot \phi(\Delta t_i)$$ | Captures temporal rhythm of user behavior |
| Temporal attention bias | $$\alpha_{ij} += \beta \cdot f(\Delta t_{ij})$$ | Recent interactions contribute more to attention |
| Long-term pooling | Learned weighted sum over transformer output | Compresses full history into stable preference |
| Local short-term window | Attend only over last $$k$$ items | Captures session-level intent shifts |
| Recency boost | $$\gamma_i \propto \exp(-\Delta t / \tau)$$ | Most recent clicks dominate short-term signal |
| Gated fusion | $$\sigma(W[s_{long} \| s_{short}])$$ | Adaptively balances exploration vs. exploitation |

## 5. Complexity Analysis

- **Long-term attention**: $$O(n^2 \cdot d)$$ per layer (standard transformer)
- **Short-term attention**: $$O(k \cdot d)$$ where $$k \ll n$$
- **Total**: $$O(L \cdot n^2 \cdot d + k \cdot d)$$ for $$L$$ layers
- **Parameters**: $$O(L \cdot d^2 + |V| \cdot d)$$ dominated by embedding table

## References

1. TLSAN original paper: Time-aware Long- and Short-term Attention Network for Next-item Recommendation (Neurocomputing, 2021)
2. SASRec: Self-Attentive Sequential Recommendation (Kang & McAuley, ICDM 2018)
3. TiSASRec: Time Interval Aware Self-Attention for Sequential Recommendation (Li et al., WSDM 2020)